# MotorAssistEnv — GRPO Training on Colab

**OpenEnv Hackathon Submission** | [HF Space](https://huggingface.co/spaces/virustechhacks/parkinsons_Motor) | [GitHub](https://github.com/your-handle/parkinson-disease)

Train a closed-loop adaptive Deep Brain Stimulation (aDBS) agent using **GRPO** with HF TRL + Unsloth 4-bit + LoRA. The agent acts as a real-time DBS programmer for a Parkinson's patient simulated by the peer-reviewed **Fleming et al. (2023)** biophysical model — observing brain biomarkers every 20 ms and tuning amplitude / pulse-width / frequency to suppress pathological beta and tremor while staying inside a clinical safety budget.

| Component | Detail |
|-----------|--------|
| Environment | [HF Space](https://huggingface.co/spaces/virustechhacks/parkinsons_Motor) — calibrated Fleming biophysics, 10 tasks |
| Training   | This Colab notebook (T4 / L4 / A100 — auto-detects) |
| Model      | `unsloth/Qwen3-4B` + LoRA (16-rank, 4-bit base) |
| Algorithm  | HF TRL `GRPOTrainer` + multi-turn rollouts |
| Reward     | Episode `grader_score` from the 9-component grader + dense per-step bonus + format compliance |
| Curriculum | `easy` (smoke test, 36 steps) → `medium` (rescue, 60 steps) → `hard` (refractory + crises, 30-step capped on T4) |

All training logic — system prompt, rollout, reward, plotting, eval — lives in **`parkinsons_Motor.train`** and is imported here exactly the way the OpenEnv-Hackathon-SF winners did it ([sid-rp/kube-sre-gym](https://github.com/sid-rp/kube-sre-gym/blob/main/colab_train_kube_.ipynb), [mhtruong1031/OpenENV-Hackathon](https://github.com/mhtruong1031/OpenENV-Hackathon)). The notebook is just glue: install → configure → import → train → plot → evaluate → push.

**Runtime:** Colab → Runtime → Change runtime type → **GPU**. Defaults complete in **30–60 min on a T4**.

See [`README.md`](./README.md) for the environment story / judging-criteria mapping, and [`REWARD_DESIGN.md`](./REWARD_DESIGN.md) for the reward-shaping rationale.

## 1 · Detect runtime + install dependencies

In [ ]:
import os, subprocess, sys, pathlib

ON_COLAB  = 'COLAB_GPU' in os.environ or os.path.isdir('/content')
ON_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or os.path.isdir('/kaggle')
BASE_DIR  = pathlib.Path('/kaggle/working') if ON_KAGGLE else (pathlib.Path('/content') if ON_COLAB else pathlib.Path.cwd())
BASE_DIR.mkdir(parents=True, exist_ok=True)
RUNTIME = 'kaggle' if ON_KAGGLE else ('colab' if ON_COLAB else 'local')
print(f'Runtime: {RUNTIME}    BASE_DIR: {BASE_DIR}')

try:
    print(subprocess.check_output(['nvidia-smi']).decode().split('\n')[0])
except Exception:
    print('nvidia-smi not available — enable GPU in Runtime settings.')

In [ ]:
%%capture
# --- Unsloth (pulls compatible torch / transformers / trl / peft / bitsandbytes) ---
!pip install -q --upgrade uv
!uv pip install -qqq --system \
    "torch==2.8.0" "triton>=3.3.0" torchvision bitsandbytes "xformers==0.0.32.post2" \
    "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
    "unsloth[base] @ git+https://github.com/unslothai/unsloth"

# --- TRL + OpenEnv + scientific stack ---
!uv pip install -qqq --system \
    "openenv-core[core]>=0.2.0" \
    "trl>=0.29,<0.30" \
    "transformers>=4.57.1,<4.58" \
    "peft>=0.15,<1" \
    "accelerate>=1.13,<2" \
    "datasets" \
    "pydantic>=2,<3" \
    "scipy==1.13.1" \
    "matplotlib" "pandas" "huggingface_hub>=0.20.0"
print('Installed.')

## 2 · Hugging Face login

Set `HF_TOKEN` in **Tools → Secrets** (Colab) or **Add-ons → Secrets** (Kaggle), label exactly `HF_TOKEN`, write-scope. Needed to clone the Space repo and push the trained adapter.

In [ ]:
import os
from huggingface_hub import login, whoami

hf_token = None
for getter in (
    lambda: __import__('google.colab', fromlist=['userdata']).userdata.get('HF_TOKEN'),
    lambda: __import__('kaggle_secrets', fromlist=['UserSecretsClient']).UserSecretsClient().get_secret('HF_TOKEN'),
    lambda: os.environ.get('HF_TOKEN'),
):
    try:
        hf_token = getter()
        if hf_token:
            break
    except Exception:
        pass
if not hf_token:
    import getpass
    hf_token = getpass.getpass('Enter your Hugging Face token (write scope, hidden): ').strip()
assert hf_token, 'HF_TOKEN required.'
login(token=hf_token, add_to_git_credential=True)
os.environ['HF_TOKEN'] = hf_token
print('Logged in as:', whoami()['name'])

## 3 · Configuration

Every knob lives here. Defaults target a T4 in ~30–60 min — scale up `NUM_TRAIN_EPISODES`, `NUM_GENERATIONS`, and `MAX_TURNS_PER_TASK['hard']` for L4 / A100.

In [ ]:
# ── Environment (live HF Space) ─────────────────────────────────────────────
ENV_URL          = 'https://virustechhacks-parkinsons-motor.hf.space'
ENV_REPO_ID      = 'virustechhacks/parkinsons_Motor'   # for cloning the client package

# ── Model + adapter target ──────────────────────────────────────────────────
MODEL_ID         = 'unsloth/Qwen3-4B'
HUB_REPO         = 'your-name/dbs-grpo-qwen3-4b'       # change before push

# ── Training curriculum ─────────────────────────────────────────────────────
TRAIN_TASKS      = ['easy', 'medium', 'hard']
EVAL_TASKS       = ['easy', 'medium', 'hard']
TRAIN_SEED_RANGE = list(range(0, 24))
MAX_TURNS_PER_TASK = {'easy': 36, 'medium': 60, 'hard': 30}

# ── GRPO hyperparameters ────────────────────────────────────────────────────
NUM_TRAIN_EPISODES         = 48
NUM_GENERATIONS            = 6      # group size; 4 is the floor for advantage stats, 6-8 ideal
PER_DEVICE_TRAIN_BATCH     = 1
GRAD_ACCUM_STEPS           = 4
LEARNING_RATE              = 2e-6
NUM_TRAIN_EPOCHS           = 1
GRPO_BETA                  = 0.02   # KL coefficient — small so the policy can move

# ── Sequence lengths ────────────────────────────────────────────────────────
# Qwen3 has chain-of-thought ON by default: completions need room for
# <think>...</think> + the final JSON action. 96 tokens stalls the model
# mid-thinking and collapses GRPO (every group gets identical rewards).
MAX_PROMPT_LENGTH          = 1280
MAX_COMPLETION_LENGTH      = 1024
MAX_SEQ_LENGTH             = 4096

# ── LoRA ────────────────────────────────────────────────────────────────────
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.0
LORA_TARGETS = ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']

# ── Sampling during rollout ─────────────────────────────────────────────────
ROLLOUT_TEMPERATURE = 0.9     # higher = more group diversity = better GRPO advantage
EVAL_TEMPERATURE    = 0.0     # deterministic at eval time for reproducible scores
EVAL_SEEDS          = [101, 202, 303, 404, 505]

SEED = 42
OUTPUT_DIR = BASE_DIR / 'artifacts' / 'motorassist-grpo'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Env:    {ENV_URL}')
print(f'Model:  {MODEL_ID}')
print(f'Output: {OUTPUT_DIR}')

## 4 · Clone the env client + smoke-test the live Space

We clone the Space repo only for the typed dataclasses (`ParkinsonsMotorAction`, `ParkinsonsMotorObservation`) and the `parkinsons_Motor.train` helpers. The actual environment runs server-side on the HF Space.

In [ ]:
import sys, subprocess

ENV_CLONE_DIR = str(BASE_DIR / 'parkinsons_motor_space')
if not os.path.isdir(ENV_CLONE_DIR):
    print(f'Cloning {ENV_REPO_ID} -> {ENV_CLONE_DIR} ...')
    rc = subprocess.call(['git', 'clone', '--depth', '1',
                          f'https://huggingface.co/spaces/{ENV_REPO_ID}', ENV_CLONE_DIR])
    assert rc == 0, 'git clone failed (check ENV_REPO_ID / network).'
else:
    print(f'Already cloned: {ENV_CLONE_DIR}')
if ENV_CLONE_DIR not in sys.path:
    sys.path.insert(0, ENV_CLONE_DIR)
print('Top-level files:', sorted(os.listdir(ENV_CLONE_DIR))[:10])

In [ ]:
import asyncio
from parkinsons_Motor import ParkinsonsMotorAction, ParkinsonsMotorEnv

async def smoke():
    env = ParkinsonsMotorEnv(base_url=ENV_URL); await env.__aenter__()
    try:
        r = await env.reset(task_id='easy', seed=0); o = r.observation
        print(f'reset OK   beta={o.beta_arv:.3f} tremor={o.tremor_arv:.3f} force={o.force_preserved:.3f}')
        r = await env.step(ParkinsonsMotorAction(motor_command=o.target_output, dbs_amplitude=1.0,
                                                  dbs_pulse_width=0.13, dbs_frequency=130.0))
        print(f'step OK    reward={r.reward:+.3f} done={r.done}')
    finally:
        await env.__aexit__(None, None, None)

asyncio.run(smoke())

## 5 · Import all training utilities from `parkinsons_Motor.train`

This is the equivalent of:

```python
# kube-sre-gym (winner)
from kube_sre_gym.train import (
    SYSTEM_PROMPT, rollout_once, format_observation, parse_commands,
    apply_chat_template, reward_total, reward_diagnosis, reward_fix,
    plot_rewards, patch_trl_vllm_compat,
)

# bio-experiment env (winner)
from training_script import (
    INVALID_ACTION_PENALTY, ENVIRONMENT_ERROR_PENALTY, OpenEnvReward,
    build_training_prompt, build_experiment_action, decode_history_actions,
    pick_action, save_training_plots,
)
```

All of our equivalents are exposed through one importable surface — see `parkinsons_Motor/train.py`.

In [ ]:
from parkinsons_Motor.train import (
    # constants
    SYSTEM_PROMPT,
    TASK_CONTEXT,
    INVALID_ACTION_PENALTY,
    ENVIRONMENT_ERROR_PENALTY,
    DEFAULT_REWARD_WEIGHTS,
    # prompt / chat
    build_user_prompt,
    apply_chat_template,
    # actions
    parse_action,
    make_action,
    heuristic_action,
    # generation + rollout
    llm_generate,
    rollout_episode,
    rollout_episode_async,
    Trajectory,
    # reward
    compute_reward,
    MotorAssistReward,
    reward_total,
    reward_grader,
    reward_dense,
    reward_format,
    # GRPO glue
    make_rollout_func,
    make_episode_logger,
    # plots
    plot_training_dashboard,
    plot_training_loss,
    plot_baseline_vs_trained,
    compare_trajectories,
    save_training_plots,
    # eval
    evaluate_model_on_task,
    evaluate_model_suite,
    eval_with_adapter_disabled,
    sanity_check_rollout,
)

print('System prompt (first 200 chars):')
print(SYSTEM_PROMPT[:200])
print('...')
print('Reward weights:', DEFAULT_REWARD_WEIGHTS)
print('Tasks supported in TASK_CONTEXT:', list(TASK_CONTEXT.keys()))

## 6 · Build the prompt dataset (one row = one episode)

Round-robin across `easy → medium → hard`, sampling random seeds. The actual `(task_id, seed)` schedule is also passed to `make_rollout_func` below so each GRPO group of size `NUM_GENERATIONS` shares identical env conditions (the kube-sre-gym trick that makes group-relative advantages clean).

In [ ]:
import random
from datasets import Dataset

rng = random.Random(SEED)
schedule = []
for i in range(NUM_TRAIN_EPISODES):
    task_id = TRAIN_TASKS[i % len(TRAIN_TASKS)]
    seed    = rng.choice(TRAIN_SEED_RANGE)
    schedule.append((task_id, seed))
rng.shuffle(schedule)

train_dataset = Dataset.from_list([
    {'prompt': f'Manage DBS for a Parkinson\'s patient on task `{tid}` (seed={s}).',
     'task_id': tid, 'seed': s}
    for tid, s in schedule
])
print(f'Train dataset: {len(train_dataset)} episodes')
print('First 5:', schedule[:5])

## 7 · Load model + LoRA via Unsloth

Qwen3-4B in 4-bit fits on a T4. LoRA is applied to attention + MLP projections.

In [ ]:
import unsloth  # must come before transformers / trl
import torch
from unsloth import FastLanguageModel, PatchFastRL

PatchFastRL('GRPO', FastLanguageModel)

bf16  = bool(getattr(torch.cuda, 'is_bf16_supported', lambda: False)()) if torch.cuda.is_available() else False
dtype = torch.bfloat16 if bf16 else (torch.float16 if torch.cuda.is_available() else torch.float32)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = MODEL_ID,
    max_seq_length  = MAX_SEQ_LENGTH,
    dtype           = dtype,
    load_in_4bit    = True,
)
if tokenizer.pad_token is None and tokenizer.eos_token is not None:
    tokenizer.pad_token = tokenizer.eos_token

model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_R,
    target_modules = LORA_TARGETS,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = LORA_DROPOUT,
    bias           = 'none',
    use_gradient_checkpointing = True,
    random_state   = SEED,
)
print(f'Model loaded. bf16={bf16}  dtype={dtype}')

## 8 · GRPO trainer — wired with `make_rollout_func` + `make_episode_logger`

The whole training loop is now five lines: pick the run dir → make a CSV episode logger → make the rollout func → build `GRPOConfig` → instantiate `GRPOTrainer`.

In [ ]:
import logging
from datetime import datetime
from trl import GRPOConfig, GRPOTrainer

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

run_dir    = OUTPUT_DIR / f'run-{datetime.now().strftime("%Y-%m-%d_%H-%M-%S")}'
run_dir.mkdir(parents=True, exist_ok=True)
reward_csv = run_dir / 'reward_log.csv'

log_episode  = make_episode_logger(reward_csv)
rollout_func = make_rollout_func(
    env_url               = ENV_URL,
    episodes              = schedule,
    max_turns_per_task    = MAX_TURNS_PER_TASK,
    num_generations       = NUM_GENERATIONS,
    log_episode           = log_episode,
    temperature           = ROLLOUT_TEMPERATURE,
    max_new_tokens        = MAX_COMPLETION_LENGTH,
    max_prompt_length     = MAX_PROMPT_LENGTH,
)

grpo_config = GRPOConfig(
    output_dir                  = str(run_dir),
    num_train_epochs            = NUM_TRAIN_EPOCHS,
    learning_rate               = LEARNING_RATE,
    per_device_train_batch_size = PER_DEVICE_TRAIN_BATCH,
    gradient_accumulation_steps = GRAD_ACCUM_STEPS,
    num_generations             = NUM_GENERATIONS,
    max_prompt_length           = MAX_PROMPT_LENGTH,
    max_completion_length       = MAX_COMPLETION_LENGTH,
    temperature                 = ROLLOUT_TEMPERATURE,    # exploration during rollouts
    top_p                       = 0.95,
    beta                        = GRPO_BETA,              # KL anchor — small for free movement
    scale_rewards               = True,                   # divide advantages by group std
    logging_steps               = 1,
    save_strategy               = 'steps',
    save_steps                  = 10,
    bf16                        = bf16,
    fp16                        = torch.cuda.is_available() and not bf16,
    report_to                   = 'none',
    remove_unused_columns       = False,
    save_total_limit            = 2,
)

trainer = GRPOTrainer(
    model            = model,
    processing_class = tokenizer,
    reward_funcs     = [reward_total, reward_grader, reward_dense, reward_format],
    args             = grpo_config,
    train_dataset    = train_dataset,
    rollout_func     = rollout_func,
)
for attr in ('image_token_id', 'vision_start_token_id', 'vision_end_token_id'):
    if not hasattr(trainer, attr):
        setattr(trainer, attr, None)
print(f'Trainer ready. CSV log → {reward_csv}')

## 9 · Pre-train sanity check — fail FAST, not silently

GRPO has two notorious failure modes that look the same in TRL's progress table (loss = 0, all rewards = 0, std = 0) but have totally different causes:

1. **Generation collapse** — every rollout emits identical actions (cap hit mid-thinking → default fallback action → identical reward → group std = 0 → no gradient).
2. **Transport collapse** — the env's persistent WebSocket times out during the (long) LLM generation because `model.generate()` blocks the asyncio event loop and the keepalive PONG never fires (close code 1011). `rollout_episode_async` now hands generation to `asyncio.to_thread(...)` so the loop stays alive.

The next two cells:

- **9a — Wake up the Space.** HF Spaces sleep after inactivity; the first WebSocket frame to a cold Space sometimes 503s. A simple HTTP GET nudges it awake before we open the persistent connection.
- **9b — Run one short rollout** and assert that JSON parses, the env returns rewards, and no exceptions fire. Raises on failure so you stop before wasting 90 minutes of GPU time.

In [ ]:
# 9a · Wake up the HF Space (HTTP nudge before the WebSocket sanity check)
import time as _time
import urllib.request as _urlreq
import urllib.error as _urlerr

def _wake_space(url: str, max_wait_s: int = 90, poll_s: float = 3.0) -> None:
    print(f'[wake-up] pinging {url} (cold Spaces can take ~60 s) ...')
    deadline = _time.time() + max_wait_s
    last_err = None
    while _time.time() < deadline:
        try:
            req = _urlreq.Request(url, headers={'User-Agent': 'motorassist-warmup/1.0'})
            with _urlreq.urlopen(req, timeout=10) as resp:
                code = resp.getcode()
                if 200 <= code < 500:
                    print(f'[wake-up] OK  HTTP {code}')
                    return
                last_err = f'HTTP {code}'
        except _urlerr.HTTPError as e:
            if 200 <= e.code < 500:
                print(f'[wake-up] OK  HTTP {e.code} (Space accepts requests)')
                return
            last_err = f'HTTPError {e.code}'
        except Exception as e:
            last_err = repr(e)
        print(f'[wake-up]   ... still waking ({last_err}); retry in {poll_s:.0f}s')
        _time.sleep(poll_s)
    print(f'[wake-up] WARNING: timed out after {max_wait_s}s; sanity check will still try '
          f'(last error: {last_err})')

_wake_space(ENV_URL)

In [ ]:
# 9b · Verify LLM produces JSON, env returns rewards, no WebSocket timeouts
#
# IMPORTANT: use the SAME max_new_tokens as training (MAX_COMPLETION_LENGTH).
# Qwen3-4B with thinking ON regularly burns 400-700 tokens before emitting
# the final JSON action; if the sanity check uses a smaller budget than
# training it will falsely reject perfectly working setups (the symptom is
# "parseable JSON 0/4" while training would actually succeed).
#
# Cost: ~3-5 min on a T4 for 4 turns. Worth it — this is the one check
# that catches all four GRPO-collapse modes before they cost you 90 min.
_ = sanity_check_rollout(
    model, tokenizer, ENV_URL,
    task_id            = 'easy',
    seed               = 0,
    max_turns          = 4,
    temperature        = ROLLOUT_TEMPERATURE,
    max_new_tokens     = MAX_COMPLETION_LENGTH,   # match training exactly
    max_prompt_length  = MAX_PROMPT_LENGTH,
    warm_up            = True,                    # compile CUDA kernels first
    retry_on_env_error = True,                    # one retry for cold Space
    raise_on_failure   = True,
)

## 10 · Train

In [ ]:
print('Starting GRPO training ...')
print(f'  tasks       : {TRAIN_TASKS}')
print(f'  episodes    : {NUM_TRAIN_EPISODES}')
print(f'  generations : {NUM_GENERATIONS} per prompt (group size)')
print(f'  env URL     : {ENV_URL}')
print()
trainer.train()
trainer.save_model(str(run_dir))
tokenizer.save_pretrained(str(run_dir))
print(f'Saved adapter + tokenizer to {run_dir}')

## 11 · Reward & loss curves

`save_training_plots` writes two figures the judges look for:

- **`training_dashboard.png`** — total reward / grader score / per-task curves / reward decomposition, computed from `reward_log.csv` (one row per episode).
- **`training_loss.png`** — policy loss / mean reward / KL-to-reference / gradient norm, pulled directly from `trainer.state.log_history` (one row per logging step).

Both PNGs are committed to the repo and embedded in [`README.md`](./README.md) §Results.

In [ ]:
from IPython.display import Image, display

plot_paths = save_training_plots(
    reward_csv, run_dir,
    train_tasks = TRAIN_TASKS,
    log_history = trainer.state.log_history,   # adds training_loss.png
)
for name, path in plot_paths.items():
    print(f'{name}: {path}')
    display(Image(filename=str(path)))

## 12 · Evaluation — **base vs trained** on `easy / medium / hard`

Five seeds per task at `temperature=0` (deterministic, reproducible). We evaluate **the same model twice** on the **same seeds**, toggling only the LoRA adapter:

1. `eval_with_adapter_disabled` → base Qwen3-4B with LoRA off (the "before" snapshot).
2. `evaluate_model_suite`        → trained policy with LoRA active (the "after" snapshot).

This isolates the effect of GRPO from any model/seed variance, and produces the per-task delta the judges look for. Outputs:

| File                       | What it is                                                       |
|----------------------------|------------------------------------------------------------------|
| `eval_baseline.json`       | per-task `mean ± std`, `pass_rate`, `mean_amp_ma` for the base  |
| `eval_trained.json`        | same fields for the trained adapter                             |
| `eval_comparison.png`      | side-by-side bar chart with per-task threshold lines + Δ labels |

Constant-baseline reference numbers from [`TASKS.md`](./TASKS.md): `easy` 0.72–0.80 (thr 0.55), `medium` 0.47–0.52 (thr 0.52), `hard` 0.23–0.36 (thr 0.68).

In [ ]:
import json as _json

eval_kwargs = dict(
    tasks               = EVAL_TASKS,
    seeds               = EVAL_SEEDS,
    max_turns_per_task  = MAX_TURNS_PER_TASK,
    temperature         = EVAL_TEMPERATURE,
    max_new_tokens      = MAX_COMPLETION_LENGTH,
    max_prompt_length   = MAX_PROMPT_LENGTH,
)

print('Evaluating BASE model (LoRA disabled) ...')
baseline_results = eval_with_adapter_disabled(
    model, tokenizer, ENV_URL, **eval_kwargs
)

print('\nEvaluating TRAINED model (LoRA active) ...')
trained_results = evaluate_model_suite(
    model, tokenizer, ENV_URL, **eval_kwargs
)

def _summary(label, results):
    print(f'\n--- {label} ---')
    for r in results:
        print(f"  {r['task_id']:6s}  mean={r['mean_score']:.3f} ± {r['std_score']:.3f}  "
              f"pass={r['pass_rate']*100:3.0f}%  amp={r['mean_amp_ma']:.2f} mA")

_summary('BASE',    baseline_results)
_summary('TRAINED', trained_results)

print('\n--- DELTA (trained − base) ---')
base_by   = {r['task_id']: r for r in baseline_results}
for r in trained_results:
    b = base_by.get(r['task_id'], {})
    d_score = r['mean_score']    - float(b.get('mean_score', 0.0))
    d_pass  = r['pass_rate']*100 - float(b.get('pass_rate', 0.0))*100
    arrow   = '↑' if d_score > 0 else ('↓' if d_score < 0 else '·')
    print(f"  {r['task_id']:6s}  Δscore={d_score:+.3f} {arrow}   Δpass={d_pass:+5.0f} pts")

def _strip(results):
    return [{k: v for k, v in r.items() if k != 'rollouts'} for r in results]

(run_dir / 'eval_baseline.json').write_text(_json.dumps(_strip(baseline_results), indent=2))
(run_dir / 'eval_trained.json').write_text(_json.dumps(_strip(trained_results), indent=2))
print(f'\nSaved {run_dir / "eval_baseline.json"}')
print(f'Saved {run_dir / "eval_trained.json"}')

## 13 · Comparison plot — base vs trained on the **same axes**

Per the judging guide: *"If you have multiple runs (baseline vs. trained, ablations, etc.), put them on the same axes so the comparison is obvious."* This cell renders `eval_comparison.png`: grouped bars per task with std-error caps, the per-task success threshold drawn as a dotted line, and the Δ (trained − base) annotated above each pair. Drop straight into [`README.md`](./README.md) §Results.

In [ ]:
comparison_png = plot_baseline_vs_trained(
    baseline_results, trained_results,
    run_dir / 'eval_comparison.png',
)
print(f'Saved {comparison_png}')
display(Image(filename=str(comparison_png)))

## 14 · Sample trajectory — **before vs after** on the same seed

Quantitative scores tell *that* the agent improved; this plot shows *how*. We pick one fixed `(task, seed)` pair and roll it out twice — LoRA disabled, then enabled — overlaying the four physiology channels the simulator returns:

- **DBS amplitude (mA)** — how aggressively the agent stimulates,
- **β-band ARV** — the bradykinesia biomarker (lower = better motor symptoms),
- **tremor ARV** — the rest-tremor biomarker (lower = better),
- **side-effect load** — the dyskinesia / paresthesia penalty (lower = safer).

A trained policy should drive β + tremor down without blowing up side-effect load — that's the whole game.

In [ ]:
DEMO_TASK = EVAL_TASKS[0] if EVAL_TASKS else 'easy'
DEMO_SEED = int(EVAL_SEEDS[0]) if EVAL_SEEDS else 0
DEMO_TURNS = MAX_TURNS_PER_TASK.get(DEMO_TASK, 30)

print(f'Rolling out task=`{DEMO_TASK}` seed={DEMO_SEED} for {DEMO_TURNS} turns x 2 (base, trained) ...')

with model.disable_adapter():
    base_traj = rollout_episode(
        model, tokenizer, ENV_URL,
        task_id=DEMO_TASK, seed=DEMO_SEED, max_turns=DEMO_TURNS,
        temperature=EVAL_TEMPERATURE,
        max_new_tokens=MAX_COMPLETION_LENGTH,
        max_prompt_length=MAX_PROMPT_LENGTH,
    )

trained_traj = rollout_episode(
    model, tokenizer, ENV_URL,
    task_id=DEMO_TASK, seed=DEMO_SEED, max_turns=DEMO_TURNS,
    temperature=EVAL_TEMPERATURE,
    max_new_tokens=MAX_COMPLETION_LENGTH,
    max_prompt_length=MAX_PROMPT_LENGTH,
)

print(f'  base    -> grader={base_traj.grader_score:.3f}  success={base_traj.episode_success}')
print(f'  trained -> grader={trained_traj.grader_score:.3f}  success={trained_traj.episode_success}')

trajectory_png = compare_trajectories(
    base_traj, trained_traj,
    run_dir / f'trajectory_compare_{DEMO_TASK}_seed{DEMO_SEED}.png',
)
print(f'Saved {trajectory_png}')
display(Image(filename=str(trajectory_png)))

## 15 · Push to Hugging Face Hub (optional)

Uncomment after editing `HUB_REPO` in cell 7.

In [ ]:
# trainer.push_to_hub(repo_id=HUB_REPO, commit_message='Initial GRPO adapter for MotorAssistEnv')
# print(f'Pushed → https://huggingface.co/{HUB_REPO}')
print('Push step is commented out by default — uncomment when HUB_REPO is set.')

## What to commit + embed in the README

This run produces every artifact the judges scan for. Commit the four PNGs and two JSONs from `run_dir`, then embed the PNGs in [`README.md`](./README.md) §Results with a one-line caption each:

| Artifact                                          | Judging criterion it satisfies                                                                  |
|---------------------------------------------------|--------------------------------------------------------------------------------------------------|
| `training_dashboard.png`                          | **20 %** Showing improvement — total/grader/per-task reward curves                               |
| `training_loss.png`                               | **min req** "loss and reward plots from a real run" — policy loss / KL / grad-norm               |
| `eval_comparison.png`                             | **20 %** "multiple runs on the same axes" — base vs trained bars + thresholds + Δ                |
| `trajectory_compare_<task>_seed<n>.png`           | **30 %** Storytelling — qualitative before/after on amplitude / β / tremor / side-effects        |
| `eval_baseline.json` + `eval_trained.json`        | **10 %** Reward & pipeline coherence — raw numbers behind the bar chart, reproducible from CSV   |
| `reward_log.csv`                                  | Engineering — per-episode log for ablations or follow-up plots                                   |

### Iterating without touching the notebook

Every helper above is in [`parkinsons_Motor/train.py`](./parkinsons_Motor/train.py): tweak `SYSTEM_PROMPT`, `DEFAULT_REWARD_WEIGHTS`, or `heuristic_action`, re-run cells 19 → 28, and the same artifacts regenerate. To add a new task (e.g. `fragile_patient`, `medication_interaction`), append the id to `TRAIN_TASKS`/`EVAL_TASKS`, set `MAX_TURNS_PER_TASK[id]`, and the rest of the pipeline picks it up — `TASK_CONTEXT` already covers all 10 expert tasks.